# From Voice to Vision — 8. From Emotion to Art

The final stage closes the loop from perception to expression. The class-probability vector
predicted for an utterance is mapped to an artistic text prompt, in which each emotion is
associated with a characteristic palette, mood and set of visual elements, and the prompt
conditions a distilled latent diffusion model that synthesises the corresponding image.

When the prediction is ambiguous — that is, when the runner-up emotion exceeds a threshold — the
two prompts are blended, so that the artwork reflects the emotional nuance rather than a hard
decision.

In [ ]:
# Clone the project repository and install the dependencies
REPO_URL = "https://github.com/Nadaa3672/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm diffusers transformers accelerate

In [ ]:
from src import config, data_loader, features
from src.models import cnn
import numpy as np, torch, matplotlib.pyplot as plt

data_loader.download_ravdess()
df = data_loader.build_index()
device = cnn.get_device()

ckpt = config.RESULTS_DIR / "best_cnn.pt"
model, ck = cnn.load_checkpoint(ckpt, device)
print("Emotion recogniser loaded:", ck["hp"])

In [ ]:
# Load the diffusion model (SD-Turbo: high quality in a few denoising steps)
from src.art import emotion_to_image as e2i
pipe = e2i.load_pipeline()

## Emotion gallery

One test utterance per emotion is classified, and the resulting prediction conditions the
generation of an artwork. Misclassifications are shown as they are: the gallery reflects the actual
behaviour of the recogniser rather than an idealised mapping.

In [ ]:
from IPython.display import Audio, display
test_df = df[df.split == "test"].reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(18, 9.5))
for i, emo in enumerate(config.EMOTIONS):
    row = test_df[test_df.emotion == emo].iloc[0]
    probs = e2i.predict_probs_from_wav(row.path, model, ck["norm"], device)
    prompt, desc = e2i.build_prompt(probs)
    img = e2i.generate(pipe, prompt, seed=config.SEED + i)
    ax = axes[i // 4, i % 4]
    ax.imshow(img); ax.axis("off")
    ax.set_title(f"true: {emo}\npredicted: {desc}", fontsize=10)
plt.suptitle("From Voice to Vision — emotion gallery (test clips)", fontsize=15)
plt.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "07_gallery.png", dpi=150)
plt.show()

## Blended prediction

When two emotions receive comparable probability, their prompts are merged. The resulting image
encodes the model's uncertainty visually, rather than discarding it.

In [ ]:
found = None
for _, row in test_df.iterrows():
    probs = e2i.predict_probs_from_wav(row.path, model, ck["norm"], device)
    if np.sort(probs)[::-1][1] >= 0.25:
        found = (row, probs)
        break

if found:
    row, probs = found
    prompt, desc = e2i.build_prompt(probs)
    img = e2i.generate(pipe, prompt, seed=config.SEED)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].bar(config.EMOTIONS, probs, color="#4C78A8")
    ax[0].set_title(f"Predicted probabilities (true label: {row.emotion})")
    ax[0].set_ylabel("Probability"); ax[0].tick_params(axis="x", rotation=40)
    ax[1].imshow(img); ax[1].axis("off")
    ax[1].set_title(f"Blended artwork: {desc}", fontsize=11)
    plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "07_blend.png", dpi=150)
    plt.show()
    display(Audio(row.path))
    print("Prompt:", prompt)

## Inference on an arbitrary recording

The same module accepts any audio file, turning a live voice into an image.

In [ ]:
from google.colab import files as gfiles
uploaded = gfiles.upload()

for fname in uploaded:
    probs = e2i.predict_probs_from_wav(fname, model, ck["norm"], device)
    prompt, desc = e2i.build_prompt(probs)
    img = e2i.generate(pipe, prompt, seed=config.SEED)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].bar(config.EMOTIONS, probs, color="#4C78A8")
    ax[0].set_title("Emotion recognised in the recording")
    ax[0].set_ylabel("Probability"); ax[0].tick_params(axis="x", rotation=40)
    ax[1].imshow(img); ax[1].axis("off"); ax[1].set_title(desc, fontsize=12)
    plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "07_custom_voice.png", dpi=150)
    plt.show()
    display(Audio(fname))
    print("Prompt:", prompt)